<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
Supplementary code for the <a href="http://mng.bz/orYv">Build a Large Language Model From Scratch</a> book by <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>Code repository: <a href="https://github.com/rasbt/LLMs-from-scratch">https://github.com/rasbt/LLMs-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="http://mng.bz/orYv"><img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>

# Chapter 5: Pretraining on Unlabeled Data

In [2]:
from importlib.metadata import version

pkgs = ["matplotlib", 
        "numpy", 
        "tiktoken", 
        "torch",
        "tensorflow" # For OpenAI's pretrained weights
       ]
for p in pkgs:
    print(f"{p} version: {version(p)}")

matplotlib version: 3.11.0
numpy version: 2.5.1
tiktoken version: 0.13.0
torch version: 2.13.0
tensorflow version: 2.21.0


- In this chapter, we implement the training loop and code for basic model evaluation to pretrain an LLM
- At the end of this chapter, we also load openly available pretrained weights from OpenAI into our model

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch05_compressed/chapter-overview.webp" width=500px>

- The topics covered in this chapter are shown below

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch05_compressed/mental-model--0.webp" width=400px>

## 5.1 Evaluating generative text models

- We start this section with a brief recap of initializing a GPT model using the code from the previous chapter
- Then, we discuss basic evaluation metrics for LLMs
- Lastly, in this section, we apply these evaluation metrics to a training and validation dataset

### 5.1.1 Using GPT to generate text

- We initialize a GPT model using the code from the previous chapter

In [3]:
# 导入 PyTorch。
# 本节使用 PyTorch 创建张量、设置随机种子、执行模型推理和计算损失。
import torch

# 从第 4 章整理出的 chapter04.py 文件中导入 GPTModel。
# GPTModel 是我们前面从零实现的完整 GPT 模型。
from previous_chapters import GPTModel

GPT_CONFIG_124M = {
    # 词汇表大小。
    # GPT-2 使用的 BPE 分词器共有 50257 个词元。
    # 因此模型最终会为每个位置输出 50257 个候选词元的分数。

    "vocab_size": 50257,   # Vocabulary size

    # 模型一次最多处理 256 个词元。
    # 原始 GPT-2 小模型的上下文长度通常是 1024。
    # 这里缩短为 256，主要是为了减少训练时间和内存占用。
    "context_length": 256, # Shortened context length (orig: 1024)

    # 每个词元的嵌入维度为 768。
    # 模型内部每个词元都会被表示成一个长度为 768 的向量。
    "emb_dim": 768,        # Embedding dimension

    # 多头注意力机制包含 12 个注意力头。
    # 768 ÷ 12 = 64，因此每个注意力头处理 64 维数据。
    "n_heads": 12,         # Number of attention heads

    # GPT 模型内部堆叠 12 个 Transformer 块。
    "n_layers": 12,        # Number of layers

    # dropout 概率为 0.1。
    # 训练期间会随机丢弃约 10% 的中间激活值，以降低过拟合风险。
    # 在 model.eval() 模式下，dropout 会被关闭。
    "drop_rate": 0.1,      # Dropout rate

     # 创建查询、键和值的线性层时不使用偏置项 bias。
    "qkv_bias": False      # Query-key-value bias
}

# 设置 PyTorch 的随机种子。
# 这样模型的随机初始化结果可以重复，便于复现实验结果。
torch.manual_seed(123)

# 根据上面定义的配置创建 GPT 模型。
# 此时模型参数只是随机初始化的，还没有经过任何训练。
model = GPTModel(GPT_CONFIG_124M)

# 将模型切换到评估模式。
# 评估模式会关闭 dropout 等只在训练阶段使用的功能。
# 注意：model.eval() 不会关闭梯度计算。
# 如果还想关闭梯度记录，需要另外使用 torch.no_grad()。
model.eval();  # Disable dropout during inference

- We use dropout of 0.1 above, but it's relatively common to train LLMs without dropout nowadays
- Modern LLMs also don't use bias vectors in the `nn.Linear` layers for the query, key, and value matrices (unlike earlier GPT models), which is achieved by setting `"qkv_bias": False`
- We reduce the context length (`context_length`) of only 256 tokens to reduce the computational resource requirements for training the model, whereas the original 124 million parameter GPT-2 model used 1024 tokens
  - This is so that more readers will be able to follow and execute the code examples on their laptop computer
  - However, please feel free to increase the `context_length` to 1024 tokens (this would not require any code changes)
  - We will also load a model with a 1024 `context_length` later from pretrained weights

- Next, we use the `generate_text_simple` function from the previous chapter to generate text
- In addition, we define two convenience functions, `text_to_token_ids` and `token_ids_to_text`, for converting between token and text representations that we use throughout this chapter

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch05_compressed/gpt-process.webp" width=500px>

In [4]:
# 导入 OpenAI GPT-2 使用的 BPE 分词器实现。
import tiktoken

# 从第 4 章导入简单的文本生成函数。
# 它会循环调用 GPT 模型，一次生成一个新词元。
from previous_chapters import generate_text_simple

# 文本 → 词元 ID 张量
# 定义一个函数，将普通字符串转换为模型能够接收的词元 ID 张量。
    #
    # text：
    #     待编码的普通文本。
    #
    # tokenizer：
    #     使用的分词器对象。
def text_to_token_ids(text, tokenizer):
    encoded = tokenizer.encode(text, allowed_special={'<|endoftext|>'})

    # 使用分词器将文本编码成词元 ID 列表。
    #
    # 例如：
    # "Every effort"
    # 可能会被转换成：
    # [6109, 3626]
    #
    # allowed_special 表示允许文本中出现特殊词元：
    # <|endoftext|>

    # 将 Python 列表转换成 PyTorch 张量。
    #
    # 假设 encoded 原来的形状为：
    # [序列长度]
    #
    # 调用 unsqueeze(0) 后变成：
    # [1, 序列长度]
    #
    # 新增加的第 0 维是批次维度。
    # 这里的 1 表示当前只有一个文本样本。
    encoded_tensor = torch.tensor(encoded).unsqueeze(0) # add batch dimension

    # 返回可以直接输入 GPT 模型的二维词元 ID 张量。
    return encoded_tensor

# 词元 ID 张量 → 文本
# 定义一个函数，将词元 ID 张量重新转换成人类可读的文本。
def token_ids_to_text(token_ids, tokenizer):
    # 删除大小为 1 的批次维度。
    #
    # 原来的形状可能为：
    # [1, 序列长度]
    #
    # 删除后变成：
    # [序列长度]
    flat = token_ids.squeeze(0) # remove batch dimension

    # 先使用 tolist() 将 PyTorch 张量转换成 Python 列表。
    # 再调用 tokenizer.decode() 将词元 ID 解码成字符串。
    # 最后返回解码后的文本。
    return tokenizer.decode(flat.tolist())

# 定义模型开始生成文本时使用的提示词。
# 它也称为初始上下文或 prompt。
start_context = "Every effort moves you"

# 创建 GPT-2 使用的 BPE 分词器。
# 该分词器的词汇表大小为 50257。
tokenizer = tiktoken.get_encoding("gpt2")

token_ids = generate_text_simple(
    # 指定用于生成文本的 GPT 模型。
    model=model,
    # 将起始文本转换成词元 ID 张量。
    # idx 的典型形状为：
    # [批次大小, 当前序列长度]
    idx=text_to_token_ids(start_context, tokenizer),
    # 最多生成 10 个新词元。
    # 这 10 个词元不包括原始提示词中的词元。
    max_new_tokens=10,
    # 模型一次最多保留多少个上下文词元。
    # 当前配置中等于 256。
    context_size=GPT_CONFIG_124M["context_length"]
)

# 将模型返回的词元 ID 转换成文本并打印。
# GPT 架构已经完成
# 但模型参数仍然是随机数
# 模型还没有学习到语言中的词序、语法和语义规律。
print("Output text:\n", token_ids_to_text(token_ids, tokenizer))

Output text:
 Every effort moves you rentingetic wasnم refres RexMeCHicular stren


- As we can see above, the model does not produce good text because it has not been trained yet
- How do we measure or capture what "good text" is, in a numeric form, to track it during training?
- The next subsection introduces metrics to calculate a loss metric for the generated outputs that we can use to measure the training progress
- The next chapters on finetuning LLMs will also introduce additional ways to measure model quality

<br>

### 5.1.2 Calculating the text generation loss: cross-entropy and perplexity

- Suppose we have an `inputs` tensor containing the token IDs for 2 training examples (rows)
- Corresponding to the `inputs`, the `targets` contain the desired token IDs that we want the model to generate
- Notice that the `targets` are the `inputs` shifted by 1 position, as explained in chapter 2 when we implemented the data loader

In [5]:
# inputs 是一个二维张量。
#
# 形状为：
# [2, 3]
#
# 2 表示批次中有两个文本样本。
# 3 表示每个样本包含三个词元。
#  
# 第一个输入样本，对应：
 # ["every", " effort", " moves"]
 # 第二个输入样本，对应：
 # ["I", " really", " like"]
inputs = torch.tensor([[16833, 3626, 6100],   # ["every effort moves",
                       [40,    1107, 588]])   #  "I really like"]

# targets 也是一个二维张量，形状为 [2, 3]。
#
# targets 相当于 inputs 整体向左移动一个位置：
#
# 输入：
# every    effort    moves
#
# 目标：
# effort   moves     you

 # 第一个目标样本，对应：
 # [" effort", " moves", " you"]
 # 第二个目标样本，对应：
 # [" really", " like", " chocolate"]
targets = torch.tensor([[3626, 6100, 345  ],  # [" effort moves you",
                        [1107,  588, 11311]]) #  " really like chocolate"]

- Feeding the `inputs` to the model, we obtain the logits vector for the 2 input examples that consist of 3 tokens each
- Each of the tokens is a 50,257-dimensional vector corresponding to the size of the vocabulary
- Applying the softmax function, we can turn the logits tensor into a tensor of the same dimension containing probability scores 

In [ ]:
# 计算文本生成损失
# 把“模型生成得不好”转换成一个可以计算的数字。
# 模型的训练目标是：
# 根据当前词元，预测下一个词元

# 临时关闭梯度计算。
    #
    # 当前只是在评估模型，还没有开始训练，
    # 因此不需要保存反向传播所需的计算图。
    #
    # 这样可以减少内存占用并提高运行速度。

with torch.no_grad():

    # 将两个输入样本传入 GPT 模型。
    #
    # logits 的形状为：
    # [批次大小, 序列长度, 词汇表大小]
    #
    # 当前为：
    # [2, 3, 50257]
    logits = model(inputs)

# 在最后一个维度，也就是词汇表维度上执行 softmax。
#
# logits 原本是任意实数分数。
# softmax 将其转换成概率：
#
# 每个概率都在 0～1 之间；
# 每个位置上的 50257 个概率总和为 1。

probas = torch.softmax(logits, dim=-1) # Probability of each token in vocabulary

# 打印概率张量的形状。
print(probas.shape) # Shape: (batch_size, num_tokens, vocab_size)

# 三个维度的含义是：
# 2      ：批次中有两个文本样本
# 3      ：每个样本包含三个位置
# 50257  ：每个位置都有 50257 个候选词元
# 模型对批次中的每一个位置，都做了一道包含 50257 个选项的选择题。

torch.Size([2, 3, 50257])


- The figure below, using a very small vocabulary for illustration purposes, outlines how we convert the probability scores back into text, which we discussed at the end of the previous chapter

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch05_compressed/proba-to-text.webp" width=500px>

- As discussed in the previous chapter, we can apply the `argmax` function to convert the probability scores into predicted token IDs
- The softmax function above produced a 50,257-dimensional vector for each token; the `argmax` function returns the position of the highest probability score in this vector, which is the predicted token ID for the given token

- Since we have 2 input batches with 3 tokens each, we obtain 2 by 3 predicted token IDs:

In [ ]:
# 使用 argmax 从 50257 个候选词元中选出概率最大的那个词元。
# # 在概率张量中寻找最大值对应的位置。
 # 在最后一个维度，也就是词汇表维度上寻找最大值。
    #
    # 不会在批次维度或序列维度上寻找。
# 保留最后一个维度。
    #
    # 不保留时，结果形状为：
    # [2, 3]
    #
    # 保留后，结果形状为：
    # [2, 3, 1]
token_ids = torch.argmax(probas, dim=-1, keepdim=True)
# 打印模型预测出的词元 ID。
print("Token IDs:\n", token_ids)

# 这里每个位置只剩下一个词元 ID，因为 argmax 已经从 50257 个候选项中选出了一个最大值。

Token IDs:
 tensor([[[16657],
         [  339],
         [42826]],

        [[49906],
         [29669],
         [41751]]])


- If we decode these tokens, we find that these are quite different from the tokens we want the model to predict, namely the target tokens:

In [ ]:
# 把目标词元和预测词元转换成文本
# targets[0] 取出第一个样本的正确目标：
# [3626, 6100, 345]
#
# 然后将它解码成文本。
print(f"Targets batch 1: {token_ids_to_text(targets[0], tokenizer)}")

# token_ids[0] 取出模型对第一个样本的预测。
#
# 由于 token_ids[0] 的形状是 [3, 1]，
# 因此调用 flatten() 将其压平成 [3]。
#
# 最后将词元 ID 解码为文本。
print(f"Outputs batch 1: {token_ids_to_text(token_ids[0].flatten(), tokenizer)}")

# 这说明随机初始化的模型没有正确预测目标词元。

Targets batch 1:  effort moves you
Outputs batch 1:  Armed heNetflix


- That's because the model wasn't trained yet
- To train the model, we need to know how far it is away from the correct predictions (targets)

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch05_compressed/proba-index.webp" width=500px>

- The token probabilities corresponding to the target indices are as follows:

In [ ]:
# 提取正确目标词元的概率
# 模型虽然会为每个位置生成 50257 个概率，但计算损失时，我们主要关注：
# 模型给正确答案分配了多大概率。

# 指定当前处理批次中的第一个文本样本。
text_idx = 0
# 选择第 0 个文本样本。
# 分别选择该文本中的第 0、1、2 个位置。
# 对每个位置，选择目标词元 ID 所对应的概率。

# 这是一种 PyTorch 高级索引写法。
#
# 它实际获取的是：
#
# probas[0, 0, targets[0, 0]]
# probas[0, 1, targets[0, 1]]
# probas[0, 2, targets[0, 2]]
target_probas_1 = probas[text_idx, [0, 1, 2], targets[text_idx]]

# 打印第一个样本中三个正确目标词元的概率。
print("Text 1:", target_probas_1)

# 指定当前处理批次中的第二个文本样本。
# 选择第 1 个文本样本。
# 选择三个词元位置。
# 从每个位置的 50257 个概率中，
# 取出正确目标词元对应的概率。
text_idx = 1
target_probas_2 = probas[text_idx, [0, 1, 2], targets[text_idx]]

# 打印第二个样本中三个目标词元的概率。
print("Text 2:", target_probas_2)

# 概率非常低，因为模型还没有经过训练。

Text 1: tensor([7.4541e-05, 3.1061e-05, 1.1563e-05])
Text 2: tensor([1.0337e-05, 5.6776e-05, 4.7559e-06])


- We want to maximize all these values, bringing them close to a probability of 1
- In mathematical optimization, it is easier to maximize the logarithm of the probability score than the probability score itself; this is out of the scope of this book, but I have recorded a lecture with more details here: [L8.2 Logistic Regression Loss Function](https://www.youtube.com/watch?v=GxJe0DZvydM)

In [ ]:
# Compute logarithm of all token probabilities
# 手动计算负平均对数概率
# 合并概率并取自然对数
# # 第一个文本样本的三个正确词元概率。
# 第二个文本样本的三个正确词元概率。
# torch.cat() 将两个长度为 3 的张量拼接成长度为 6 的张量。
#
# torch.log() 对每个概率取自然对数。
#
# 由于概率都在 0～1 之间，
# 因此取对数后得到的数值都是负数。

log_probas = torch.log(torch.cat((target_probas_1, target_probas_2)))

# 打印六个正确目标词元的对数概率。
print(log_probas)

# 正确词元概率越小，对数结果就越负。
# 例如：
# 概率 0.9    → log(0.9)   ≈ -0.105
# 概率 0.1    → log(0.1)   ≈ -2.303
# 概率 0.001  → log(0.001) ≈ -6.908

tensor([ -9.5042, -10.3796, -11.3677, -11.4798,  -9.7764, -12.2561])


- Next, we compute the average log probability:

In [ ]:
# Calculate the average probability for each token
# # 计算六个正确目标词元对数概率的平均值。
avg_log_probas = torch.mean(log_probas)

# # 打印平均对数概率。
print(avg_log_probas)

# 这个值越接近 0，表示模型为正确词元分配的概率越高。

tensor(-10.7940)


- The goal is to make this average log probability as large as possible by optimizing the model weights
- Due to the log, the largest possible value is 0, and we are currently far away from 0

- In deep learning, instead of maximizing the average log-probability, it's a standard convention to minimize the *negative* average log-probability value; in our case, instead of maximizing -10.7722 so that it approaches 0, in deep learning, we would minimize 10.7722 so that it approaches 0
- The value negative of -10.7722, i.e., 10.7722, is also called cross-entropy loss in deep learning

In [ ]:
# 将平均对数概率乘以 -1。
#
# 原来的值是负数：
# -10.7940
#
# 乘以 -1 后变为：
# 10.7940
neg_avg_log_probas = avg_log_probas * -1

# 打印负平均对数概率。
print(neg_avg_log_probas)

# 训练模型时，我们的目标就是：
# 让这个损失不断减小

tensor(10.7940)


- PyTorch already implements a `cross_entropy` function that carries out the previous steps

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch05_compressed/cross-entropy.webp?123" width=400px>

- Before we apply the `cross_entropy` function, let's check the shape of the logits and targets

In [ ]:
# 使用 PyTorch 计算交叉熵损失
# 手动计算过程是：
# logits
#  ↓ softmax
# 概率
# ↓ 提取目标词元概率
# 正确答案概率
#  ↓ log
# 对数概率
#  ↓ 求负平均值
# 交叉熵损失
#
#
#
#
# PyTorch 的 cross_entropy() 可以一次完成这些步骤。
# Logits have shape (batch_size, num_tokens, vocab_size)
# 打印模型原始输出 logits 的形状。
print("Logits shape:", logits.shape)

# Targets have shape (batch_size, num_tokens)
# 打印目标词元张量的形状。
print("Targets shape:", targets.shape)

# logits：
# 2 个样本 × 3 个位置 × 50257 个候选词元

# targets：
# 2 个样本 × 3 个正确词元 ID

Logits shape: torch.Size([2, 3, 50257])
Targets shape: torch.Size([2, 3])


- For the `cross_entropy` function in PyTorch, we want to flatten these tensors by combining them over the batch dimension:

In [18]:
# 将 logits 的第 0 维和第 1 维合并。
#
# 原来的形状：
# [2, 3, 50257]
#
# 展平后的形状：
# [6, 50257]
#
# 也就是将两个样本中的六个词元位置，
# 看成六道独立的分类题。
logits_flat = logits.flatten(0, 1)
# 将目标张量完全展平。
#
# 原来的形状：
# [2, 3]
#
# 展平后的形状：
# [6]
targets_flat = targets.flatten()


# 打印展平后的模型输出形状。
print("Flattened logits:", logits_flat.shape)

# 打印展平后的目标形状。
print("Flattened targets:", targets_flat.shape)

# 可以理解为
# 一共有 6 道选择题
# 每道题有 50257 个选项
# 每道题有一个正确答案

Flattened logits: torch.Size([6, 50257])
Flattened targets: torch.Size([6])


- Note that the targets are the token IDs, which also represent the index positions in the logits tensors that we want to maximize
- The `cross_entropy` function in PyTorch will automatically take care of applying the softmax and log-probability computation internally over those token indices in the logits that are to be maximized 

In [ ]:
# 直接计算交叉熵
#  # 模型对六个位置输出的原始 logits。
    #
    # 形状：
    # [6, 50257]
# 六个位置对应的正确词元 ID。
    #
    # 形状：
    # [6]
loss = torch.nn.functional.cross_entropy(logits_flat, targets_flat)

# 打印交叉熵损失。
print(loss)

# 这与前面手动计算的负平均对数概率相同。

tensor(10.7940)


- A concept related to the cross-entropy loss is the perplexity of an LLM
- The perplexity is simply the exponential of the cross-entropy loss

In [ ]:
# 计算困惑度
# # 对交叉熵损失取指数，得到困惑度。
perplexity = torch.exp(loss)

# 打印困惑度。
print(perplexity)

# 可以直观理解成：

# 模型在预测下一个词元时，像是在大约 48726 个候选词元之间犹豫。

# 因为词汇表一共有 50257 个词元，所以这个结果说明未经训练的模型接近随机猜测。

- The perplexity is often considered more interpretable because it can be understood as the effective vocabulary size that the model is uncertain about at each step (in the example above, that'd be 47,678 words or tokens)
- In other words, perplexity provides a measure of how well the probability distribution predicted by the model matches the actual distribution of the words in the dataset
- Similar to the loss, a lower perplexity indicates that the model predictions are closer to the actual distribution

### 5.1.3 Calculating the training and validation set losses

- We use a relatively small dataset for training the LLM (in fact, only one short story)
- The reasons are:
  - You can run the code examples in a few minutes on a laptop computer without a suitable GPU
  - The training finishes relatively fast (minutes instead of weeks), which is good for educational purposes
  - We use a text from the public domain, which can be included in this GitHub repository without violating any usage rights or bloating the repository size


- For example, Llama 2 7B required 184,320 GPU hours on A100 GPUs to be trained on 2 trillion tokens
  - At the time of this writing, the hourly cost of an 8xA100 cloud server at AWS is approximately \\$30
  - So, via an off-the-envelope calculation, training this LLM would cost 184,320 / 8 * \\$30 =  \\$690,000
 
- Below, we use the same dataset we used in chapter 2

In [20]:
# 读取《The Verdict》文本
# 整体功能

# 这段代码读取训练文本，并统计字符数量和词元数量。
import os
import urllib.request

# 指定训练文本文件的路径。
# 默认文件位于当前 Python 程序的工作目录下。
file_path = "the-verdict.txt"
url = "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/main/ch02/01_main-chapter-code/the-verdict.txt"

if not os.path.exists(file_path):
    with urllib.request.urlopen(url) as response:
        text_data = response.read().decode('utf-8')
    with open(file_path, "w", encoding="utf-8") as file:
        # 以只读模式打开文本文件。
    #
    # encoding="utf-8" 指定使用 UTF-8 编码读取。
        file.write(text_data)
else:
    with open(file_path, "r", encoding="utf-8") as file:
        # 一次性读取文件中的所有内容，
    # 并保存到字符串变量 text_data 中。
        text_data = file.read()

- A quick check that the text loaded ok by printing the first and last 100 words

In [21]:
# First 100 characters
print(text_data[:99])

I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no 


In [22]:
# Last 100 characters
print(text_data[-99:])

it for me! The Strouds stand alone, and happen once--but there's no exterminating our kind of art."


In [ ]:
# 计算原始字符串中的字符总数。
#
# 空格、标点符号和换行符也会计入字符数量。
total_characters = len(text_data)

# 使用 GPT-2 分词器对全部文本进行编码。
# 然后统计编码后包含多少个词元。
total_tokens = len(tokenizer.encode(text_data))

# 打印字符总数。
print("Characters:", total_characters)

# 打印词元总数。
print("Tokens:", total_tokens)

# 字符数量 ≠ 词元数量
# 因为一个词元可能包含：

# 一个完整单词；
# 一个子词；
# 标点符号；
# 单词的一部分；
# 空格和单词的组合。

Characters: 20479
Tokens: 5145


- With 5,145 tokens, the text is very short for training an LLM, but again, it's for educational purposes (we will also load pretrained weights later)

- Next, we divide the dataset into a training and a validation set and use the data loaders from chapter 2 to prepare the batches for LLM training
- For visualization purposes, the figure below assumes a `max_length=6`, but for the training loader, we set the `max_length` equal to the context length that the LLM supports
- The figure below only shows the input tokens for simplicity
    - Since we train the LLM to predict the next word in the text, the targets look the same as these inputs, except that the targets are shifted by one position

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch05_compressed/batching.webp" width=500px>

In [24]:
from previous_chapters import create_dataloader_v1

# Train/validation ratio
# 指定训练集占整个文本的 90%。
train_ratio = 0.90

# 计算训练集和验证集的分割位置。
#
# len(text_data) 是字符总数。
# 乘以 0.90 后得到前 90% 字符的位置。
#
# int() 将结果转换成整数索引。
split_idx = int(train_ratio * len(text_data))

# 从文本开头截取到 split_idx，
# 作为训练数据。
train_data = text_data[:split_idx]

# 从 split_idx 截取到文本末尾，
# 作为验证数据。
val_data = text_data[split_idx:]

# 这里是按照字符位置划分，而不是按照词元位置划分。
# 最终关系大致为：
# 完整文本
# ├── 前 90%：train_data
# └── 后 10%：val_data

# 设置随机种子。
# 因为训练集会执行 shuffle=True，
# 设置随机种子有助于复现相同的打乱顺序。
torch.manual_seed(123)

# 从第 2 章导入创建 GPT 数据加载器的函数。
#
# 该函数负责：
# 1. 对文本进行分词；
# 2. 使用滑动窗口切分文本；
# 3. 创建输入和目标；
# 4. 按批次返回数据。
train_loader = create_dataloader_v1(
    # 传入训练文本。
    train_data,

    # 每个批次包含两个文本样本。
    #
    # 真实大模型训练一般会使用更大的有效批次大小，
    # 这里使用 2 是为了降低教学实验的计算需求。
    batch_size=2,

    # 每个文本样本包含的最大词元数。
    #
    # 当前等于 256。
    max_length=GPT_CONFIG_124M["context_length"],

    # 滑动窗口每次向前移动 256 个词元。
    #
    # 因为 stride 与 max_length 相同，
    # 相邻文本块之间不会重叠。
    stride=GPT_CONFIG_124M["context_length"],

    # 如果最后一个批次不够两个样本，就将它丢弃。
    #
    # 训练时保持每个批次形状一致，通常更方便。
    drop_last=True,
    
    # 每轮训练时打乱样本顺序。
    #
    # 有助于减少模型对固定样本顺序的依赖。
    shuffle=True,

    # 使用主进程加载数据。
    #
    # 设为 0 兼容性最好，
    # 特别适合 Jupyter Notebook 和简单实验环境。
    num_workers=0
)

val_loader = create_dataloader_v1(
     # 传入验证文本。
    val_data,

    # 每个验证批次最多包含两个样本。
    batch_size=2,

    # 每个样本最多包含 256 个词元。
    max_length=GPT_CONFIG_124M["context_length"],

    # 每次向前移动 256 个词元，
    # 相邻样本之间不重叠。
    stride=GPT_CONFIG_124M["context_length"],

    # 验证时不丢弃最后一个不完整批次。
    #
    # 因为验证阶段希望尽量利用全部验证数据。
    drop_last=False,

     # 验证集不需要打乱。
    #
    # 保持固定顺序有利于得到稳定、可重复的评估结果。
    shuffle=False,

    # 在当前主进程中加载验证数据。
    num_workers=0
)

In [ ]:
# Sanity check

if total_tokens * (train_ratio) < GPT_CONFIG_124M["context_length"]:
    print("Not enough tokens for the training loader. "
          "Try to lower the `GPT_CONFIG_124M['context_length']` or "
          "increase the `training_ratio`")

if total_tokens * (1-train_ratio) < GPT_CONFIG_124M["context_length"]:
    print("Not enough tokens for the validation loader. "
          "Try to lower the `GPT_CONFIG_124M['context_length']` or "
          "decrease the `training_ratio`")

- We use a relatively small batch size to reduce the computational resource demand, and because the dataset is very small to begin with
- Llama 2 7B was trained with a batch size of 1024, for example

- An optional check that the data was loaded correctly:

In [25]:
# 打印训练集标题。
print("Train loader:")

# 遍历训练数据加载器。
    #
    # x 是输入词元 ID。
    # y 是对应的目标词元 ID。
for x, y in train_loader:

     # 打印输入和目标的形状。
    print(x.shape, y.shape)

# 先打印一个换行符，再打印验证集标题。
print("\nValidation loader:")
for x, y in val_loader:
    # 遍历验证数据加载器。
    print(x.shape, y.shape)
    # 打印每个验证批次中输入和目标的形状。

Train loader:
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])

Validation loader:
torch.Size([2, 256]) torch.Size([2, 256])


- Another optional check that the token sizes are in the expected ballpark:

In [26]:
train_tokens = 0
for input_batch, target_batch in train_loader:
    train_tokens += input_batch.numel()

val_tokens = 0
for input_batch, target_batch in val_loader:
    val_tokens += input_batch.numel()

print("Training tokens:", train_tokens)
print("Validation tokens:", val_tokens)
print("All tokens:", train_tokens + val_tokens)

Training tokens: 4608
Validation tokens: 512
All tokens: 5120


- Next, we implement a utility function to calculate the cross-entropy loss of a given batch
- In addition, we implement a second utility function to compute the loss for a user-specified number of batches in a data loader

In [ ]:
def calc_loss_batch(input_batch, target_batch, model, device):
    input_batch, target_batch = input_batch.to(device), target_batch.to(device)
    logits = model(input_batch)
    loss = torch.nn.functional.cross_entropy(logits.flatten(0, 1), target_batch.flatten())
    return loss


def calc_loss_loader(data_loader, model, device, num_batches=None):
    total_loss = 0.
    if len(data_loader) == 0:
        return float("nan")
    elif num_batches is None:
        num_batches = len(data_loader)
    else:
        # Reduce the number of batches to match the total number of batches in the data loader
        # if num_batches exceeds the number of batches in the data loader
        num_batches = min(num_batches, len(data_loader))
    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i < num_batches:
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            total_loss += loss.item()
        else:
            break
    return total_loss / num_batches

- If you have a machine with a CUDA-supported GPU, the LLM will train on the GPU without making any changes to the code
- Via the `device` setting, we ensure that the data is loaded onto the same device as the LLM model

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device) # no assignment model = model.to(device) necessary for nn.Module classes


torch.manual_seed(123) # For reproducibility due to the shuffling in the data loader

with torch.no_grad(): # Disable gradient tracking for efficiency because we are not training, yet
    train_loss = calc_loss_loader(train_loader, model, device)
    val_loss = calc_loss_loader(val_loader, model, device)

print("Training loss:", train_loss)
print("Validation loss:", val_loss)

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch05_compressed/mental-model-1.webp" width=400px>

## 5.2 Training an LLM

- In this section, we finally implement the code for training the LLM
- We focus on a simple training function (if you are interested in augmenting this training function with more advanced techniques, such as learning rate warmup, cosine annealing, and gradient clipping, please refer to [Appendix D](../../appendix-D/01_main-chapter-code))

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch05_compressed/train-steps.webp" width=300px>

In [ ]:
def train_model_simple(model, train_loader, val_loader, optimizer, device, num_epochs,
                       eval_freq, eval_iter, start_context, tokenizer):
    # Initialize lists to track losses and tokens seen
    train_losses, val_losses, track_tokens_seen = [], [], []
    tokens_seen, global_step = 0, -1

    # Main training loop
    for epoch in range(num_epochs):
        model.train()  # Set model to training mode
        
        for input_batch, target_batch in train_loader:
            optimizer.zero_grad() # Reset loss gradients from previous batch iteration
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            loss.backward() # Calculate loss gradients
            optimizer.step() # Update model weights using loss gradients
            tokens_seen += input_batch.numel()
            global_step += 1

            # Optional evaluation step
            if global_step % eval_freq == 0:
                train_loss, val_loss = evaluate_model(
                    model, train_loader, val_loader, device, eval_iter)
                train_losses.append(train_loss)
                val_losses.append(val_loss)
                track_tokens_seen.append(tokens_seen)
                print(f"Ep {epoch+1} (Step {global_step:06d}): "
                      f"Train loss {train_loss:.3f}, Val loss {val_loss:.3f}")

        # Print a sample text after each epoch
        generate_and_print_sample(
            model, tokenizer, device, start_context
        )

    return train_losses, val_losses, track_tokens_seen


def evaluate_model(model, train_loader, val_loader, device, eval_iter):
    model.eval()
    with torch.no_grad():
        train_loss = calc_loss_loader(train_loader, model, device, num_batches=eval_iter)
        val_loss = calc_loss_loader(val_loader, model, device, num_batches=eval_iter)
    model.train()
    return train_loss, val_loss


def generate_and_print_sample(model, tokenizer, device, start_context):
    model.eval()
    context_size = model.pos_emb.weight.shape[0]
    encoded = text_to_token_ids(start_context, tokenizer).to(device)
    with torch.no_grad():
        token_ids = generate_text_simple(
            model=model, idx=encoded,
            max_new_tokens=50, context_size=context_size
        )
        decoded_text = token_ids_to_text(token_ids, tokenizer)
        print(decoded_text.replace("\n", " "))  # Compact print format
    model.train()

- Now, let's train the LLM using the training function defined above:

In [ ]:
torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M)
model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.0004, weight_decay=0.1)

num_epochs = 10
train_losses, val_losses, tokens_seen = train_model_simple(
    model, train_loader, val_loader, optimizer, device,
    num_epochs=num_epochs, eval_freq=5, eval_iter=5,
    start_context="Every effort moves you", tokenizer=tokenizer
)

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator


def plot_losses(epochs_seen, tokens_seen, train_losses, val_losses):
    fig, ax1 = plt.subplots(figsize=(5, 3))

    # Plot training and validation loss against epochs
    ax1.plot(epochs_seen, train_losses, label="Training loss")
    ax1.plot(epochs_seen, val_losses, linestyle="-.", label="Validation loss")
    ax1.set_xlabel("Epochs")
    ax1.set_ylabel("Loss")
    ax1.legend(loc="upper right")
    ax1.xaxis.set_major_locator(MaxNLocator(integer=True))  # only show integer labels on x-axis

    # Create a second x-axis for tokens seen
    ax2 = ax1.twiny()  # Create a second x-axis that shares the same y-axis
    ax2.plot(tokens_seen, train_losses, alpha=0)  # Invisible plot for aligning ticks
    ax2.set_xlabel("Tokens seen")

    fig.tight_layout()  # Adjust layout to make room
    plt.savefig("loss-plot.pdf")
    plt.show()

epochs_tensor = torch.linspace(0, num_epochs, len(train_losses))
plot_losses(epochs_tensor, tokens_seen, train_losses, val_losses)

- Looking at the results above, we can see that the model starts out generating incomprehensible strings of words, whereas towards the end, it's able to produce grammatically more or less correct sentences
- However, based on the training and validation set losses, we can see that the model starts overfitting
- If we were to check a few passages it writes towards the end, we would find that they are contained in the training set verbatim -- it simply memorizes the training data
- Later, we will cover decoding strategies that can mitigate this memorization by a certain degree
- Note that the overfitting here occurs because we have a very, very small training set, and we iterate over it so many times
  - The LLM training here primarily serves educational purposes; we mainly want to see that the model can learn to produce coherent text
  - Instead of spending weeks or months on training this model on vast amounts of expensive hardware, we load pretrained weights later

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch05_compressed/mental-model-2.webp" width=350px>

**If you are interested in augmenting this training function with more advanced techniques, such as learning rate warmup, cosine annealing, and gradient clipping, please refer to [Appendix D](../../appendix-D/01_main-chapter-code)**

**If you are interested in a larger training dataset and longer training run, see [../03_bonus_pretraining_on_gutenberg](../03_bonus_pretraining_on_gutenberg)**

## 5.3 Decoding strategies to control randomness

- Inference is relatively cheap with a relatively small LLM as the GPT model we trained above, so there's no need to use a GPU for it in case you used a GPU for training it above
- Using the `generate_text_simple` function (from the previous chapter) that we used earlier inside the simple training function, we can generate new text one word (or token) at a time
- As explained in section 5.1.2, the next generated token is the token corresponding to the largest probability score among all tokens in the vocabulary

In [ ]:
model.to("cpu")
model.eval()

tokenizer = tiktoken.get_encoding("gpt2")

token_ids = generate_text_simple(
    model=model,
    idx=text_to_token_ids("Every effort moves you", tokenizer),
    max_new_tokens=25,
    context_size=GPT_CONFIG_124M["context_length"]
)

print("Output text:\n", token_ids_to_text(token_ids, tokenizer))

- Even if we execute the `generate_text_simple` function above multiple times, the LLM will always generate the same outputs
- We now introduce two concepts, so-called decoding strategies, to modify the `generate_text_simple`: *temperature scaling* and *top-k* sampling
- These will allow the model to control the randomness and diversity of the generated text

### 5.3.1 Temperature scaling

- Previously, we always sampled the token with the highest probability as the next token using `torch.argmax`
- To add variety, we can sample the next token using The `torch.multinomial(probs, num_samples=1)`, sampling from a probability distribution
- Here, each index's chance of being picked corresponds to its probability in the input tensor

- Here's a little recap of generating the next token, assuming a very small vocabulary for illustration purposes:

In [ ]:
vocab = { 
    "closer": 0,
    "every": 1, 
    "effort": 2, 
    "forward": 3,
    "inches": 4,
    "moves": 5, 
    "pizza": 6,
    "toward": 7,
    "you": 8,
} 

inverse_vocab = {v: k for k, v in vocab.items()}

# Suppose input is "every effort moves you", and the LLM
# returns the following logits for the next token:
next_token_logits = torch.tensor(
    [4.51, 0.89, -1.90, 6.75, 1.63, -1.62, -1.89, 6.28, 1.79]
)

probas = torch.softmax(next_token_logits, dim=0)
next_token_id = torch.argmax(probas).item()

# The next generated token is then as follows:
print(inverse_vocab[next_token_id])

In [ ]:
torch.manual_seed(123)
next_token_id = torch.multinomial(probas, num_samples=1).item()
print(inverse_vocab[next_token_id])

In [ ]:
def print_sampled_tokens(probas):
    torch.manual_seed(123) # Manual seed for reproducibility
    sample = [torch.multinomial(probas, num_samples=1).item() for i in range(1_000)]
    sampled_ids = torch.bincount(torch.tensor(sample))
    for i, freq in enumerate(sampled_ids):
        print(f"{freq} x {inverse_vocab[i]}")

print_sampled_tokens(probas)

- Instead of determining the most likely token via `torch.argmax`, we use `torch.multinomial(probas, num_samples=1)` to determine the most likely token by sampling from the softmax distribution
- For illustration purposes, let's see what happens when we sample the next token 1,000 times using the original softmax probabilities:

- We can control the distribution and selection process via a concept called temperature scaling
- "Temperature scaling" is just a fancy word for dividing the logits by a number greater than 0
- Temperatures greater than 1 will result in more uniformly distributed token probabilities after applying the softmax
- Temperatures smaller than 1 will result in more confident (sharper or more peaky) distributions after applying the softmax

In [ ]:
def softmax_with_temperature(logits, temperature):
    scaled_logits = logits / temperature
    return torch.softmax(scaled_logits, dim=0)

# Temperature values
temperatures = [1, 0.1, 5]  # Original, higher confidence, and lower confidence

# Calculate scaled probabilities
scaled_probas = [softmax_with_temperature(next_token_logits, T) for T in temperatures]

In [ ]:
# Plotting
x = torch.arange(len(vocab))
bar_width = 0.15

fig, ax = plt.subplots(figsize=(5, 3))
for i, T in enumerate(temperatures):
    rects = ax.bar(x + i * bar_width, scaled_probas[i], bar_width, label=f'Temperature = {T}')

ax.set_ylabel('Probability')
ax.set_xticks(x)
ax.set_xticklabels(vocab.keys(), rotation=90)
ax.legend()

plt.tight_layout()
plt.savefig("temperature-plot.pdf")
plt.show()

- We can see that the rescaling via temperature 0.1 results in a sharper distribution, approaching `torch.argmax`, such that the most likely word is almost always selected:

In [ ]:
print_sampled_tokens(scaled_probas[1])

- The rescaled probabilities via temperature 5 are more uniformly distributed:

In [ ]:
print_sampled_tokens(scaled_probas[2])

- Assuming an LLM input "every effort moves you", using the approach above can sometimes result in nonsensical texts, such as "every effort moves you pizza", 3.2% of the time (32 out of 1000 times)

### 5.3.2 Top-k sampling

- To be able to use higher temperatures to increase output diversity and to reduce the probability of nonsensical sentences, we can restrict the sampled tokens to the top-k most likely tokens:

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch05_compressed/topk.webp" width=500px>

- (Please note that the numbers in this figure are truncated to two
digits after the decimal point to reduce visual clutter. The values in the Softmax row should add up to 1.0.)

- In code, we can implement this as follows:

In [ ]:
top_k = 3
top_logits, top_pos = torch.topk(next_token_logits, top_k)

print("Top logits:", top_logits)
print("Top positions:", top_pos)

In [ ]:
new_logits = torch.where(
    condition=next_token_logits < top_logits[-1],
    input=torch.tensor(float('-inf')), 
    other=next_token_logits
)

print(new_logits)

In [ ]:
topk_probas = torch.softmax(new_logits, dim=0)
print(topk_probas)

### 5.3.3 Modifying the text generation function

- The previous two subsections introduced temperature sampling and top-k sampling
- Let's use these two concepts to modify the `generate_simple` function we used to generate text via the LLM earlier, creating a new `generate` function:

In [ ]:
def generate(model, idx, max_new_tokens, context_size, temperature=0.0, top_k=None, eos_id=None):

    # For-loop is the same as before: Get logits, and only focus on last time step
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -context_size:]
        with torch.no_grad():
            logits = model(idx_cond)
        logits = logits[:, -1, :]

        # New: Filter logits with top_k sampling
        if top_k is not None:
            # Keep only top_k values
            top_logits, _ = torch.topk(logits, top_k)
            min_val = top_logits[:, -1]
            logits = torch.where(logits < min_val, torch.tensor(float('-inf')).to(logits.device), logits)

        # New: Apply temperature scaling
        if temperature > 0.0:
            logits = logits / temperature

            # Apply softmax to get probabilities
            probs = torch.softmax(logits, dim=-1)  # (batch_size, context_len)

            # Sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1)  # (batch_size, 1)

        # Otherwise same as before: get idx of the vocab entry with the highest logits value
        else:
            idx_next = torch.argmax(logits, dim=-1, keepdim=True)  # (batch_size, 1)

        if idx_next == eos_id:  # Stop generating early if end-of-sequence token is encountered and eos_id is specified
            break

        # Same as before: append sampled index to the running sequence
        idx = torch.cat((idx, idx_next), dim=1)  # (batch_size, num_tokens+1)

    return idx

In [ ]:
torch.manual_seed(123)

token_ids = generate(
    model=model,
    idx=text_to_token_ids("Every effort moves you", tokenizer),
    max_new_tokens=15,
    context_size=GPT_CONFIG_124M["context_length"],
    top_k=25,
    temperature=1.4
)

print("Output text:\n", token_ids_to_text(token_ids, tokenizer))

## 5.4 Loading and saving model weights in PyTorch

- Training LLMs is computationally expensive, so it's crucial to be able to save and load LLM weights

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch05_compressed/mental-model-3.webp" width=400px>

- The recommended way in PyTorch is to save the model weights, the so-called `state_dict` via by applying the `torch.save` function to the `.state_dict()` method:

In [ ]:
torch.save(model.state_dict(), "model.pth")

- Then we can load the model weights into a new `GPTModel` model instance as follows:

In [ ]:
model = GPTModel(GPT_CONFIG_124M)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.load_state_dict(torch.load("model.pth", map_location=device))
model.eval();

- It's common to train LLMs with adaptive optimizers like Adam or AdamW instead of regular SGD
- These adaptive optimizers store additional parameters for each model weight, so it makes sense to save them as well in case we plan to continue the pretraining later:

In [ ]:
torch.save({
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    }, 
    "model_and_optimizer.pth"
)

In [ ]:
checkpoint = torch.load("model_and_optimizer.pth")

model = GPTModel(GPT_CONFIG_124M)
model.load_state_dict(checkpoint["model_state_dict"])

optimizer = torch.optim.AdamW(model.parameters(), lr=0.0005, weight_decay=0.1)
optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
model.train();

## 5.5 Loading pretrained weights from OpenAI

- Previously, we only trained a small GPT-2 model using a very small short-story book for educational purposes
- Interested readers can also find a longer pretraining run on the complete Project Gutenberg book corpus in [../03_bonus_pretraining_on_gutenberg](../03_bonus_pretraining_on_gutenberg)
- Fortunately, we don't have to spend tens to hundreds of thousands of dollars to pretrain the model on a large pretraining corpus but can load the pretrained weights provided by OpenAI

- For an alternative way to load the weights from the Hugging Face Hub, see [../02_alternative_weight_loading](../02_alternative_weight_loading)

- First, some boilerplate code to download the files from OpenAI and load the weights into Python
- Since OpenAI used [TensorFlow](https://www.tensorflow.org/), we will have to install and use TensorFlow for loading the weights; [tqdm](https://github.com/tqdm/tqdm) is a progress bar library
- Uncomment and run the next cell to install the required libraries

In [ ]:
# pip install tensorflow tqdm

In [ ]:
print("TensorFlow version:", version("tensorflow"))
print("tqdm version:", version("tqdm"))

In [ ]:
# Relative import from the gpt_download.py contained in this folder
from gpt_download import download_and_load_gpt2

- We can then download the model weights for the 124 million parameter model as follows:

In [ ]:
settings, params = download_and_load_gpt2(model_size="124M", models_dir="gpt2")

In [ ]:
print("Settings:", settings)

In [ ]:
print("Parameter dictionary keys:", params.keys())

In [ ]:
print(params["wte"])
print("Token embedding weight tensor dimensions:", params["wte"].shape)

- Alternatively, "355M", "774M", and "1558M" are also supported `model_size` arguments
- The difference between these differently sized models is summarized in the figure below:

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch05_compressed/gpt-sizes.webp?timestamp=123" width=500px>

- Above, we loaded the 124M GPT-2 model weights into Python, however we still need to transfer them into our `GPTModel` instance
- First, we initialize a new GPTModel instance
- Note that the original GPT model initialized the linear layers for the query, key, and value matrices in the multi-head attention module with bias vectors, which is not required or recommended; however, to be able to load the weights correctly, we have to enable these too by setting `qkv_bias` to `True` in our implementation, too
- We are also using the `1024` token context length that was used by the original GPT-2 model(s)

In [ ]:
# Define model configurations in a dictionary for compactness
model_configs = {
    "gpt2-small (124M)": {"emb_dim": 768, "n_layers": 12, "n_heads": 12},
    "gpt2-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
    "gpt2-large (774M)": {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
    "gpt2-xl (1558M)": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25},
}

# Copy the base configuration and update with specific model settings
model_name = "gpt2-small (124M)"  # Example model name
NEW_CONFIG = GPT_CONFIG_124M.copy()
NEW_CONFIG.update(model_configs[model_name])
NEW_CONFIG.update({"context_length": 1024, "qkv_bias": True})

gpt = GPTModel(NEW_CONFIG)
gpt.eval();

- The next task is to assign the OpenAI weights to the corresponding weight tensors in our `GPTModel` instance

In [ ]:
def assign(left, right):
    if left.shape != right.shape:
        raise ValueError(f"Shape mismatch. Left: {left.shape}, Right: {right.shape}")
    return torch.nn.Parameter(torch.tensor(right))

In [ ]:
import numpy as np

def load_weights_into_gpt(gpt, params):
    gpt.pos_emb.weight = assign(gpt.pos_emb.weight, params['wpe'])
    gpt.tok_emb.weight = assign(gpt.tok_emb.weight, params['wte'])
    
    for b in range(len(params["blocks"])):
        q_w, k_w, v_w = np.split(
            (params["blocks"][b]["attn"]["c_attn"])["w"], 3, axis=-1)
        gpt.trf_blocks[b].att.W_query.weight = assign(
            gpt.trf_blocks[b].att.W_query.weight, q_w.T)
        gpt.trf_blocks[b].att.W_key.weight = assign(
            gpt.trf_blocks[b].att.W_key.weight, k_w.T)
        gpt.trf_blocks[b].att.W_value.weight = assign(
            gpt.trf_blocks[b].att.W_value.weight, v_w.T)

        q_b, k_b, v_b = np.split(
            (params["blocks"][b]["attn"]["c_attn"])["b"], 3, axis=-1)
        gpt.trf_blocks[b].att.W_query.bias = assign(
            gpt.trf_blocks[b].att.W_query.bias, q_b)
        gpt.trf_blocks[b].att.W_key.bias = assign(
            gpt.trf_blocks[b].att.W_key.bias, k_b)
        gpt.trf_blocks[b].att.W_value.bias = assign(
            gpt.trf_blocks[b].att.W_value.bias, v_b)

        gpt.trf_blocks[b].att.out_proj.weight = assign(
            gpt.trf_blocks[b].att.out_proj.weight, 
            params["blocks"][b]["attn"]["c_proj"]["w"].T)
        gpt.trf_blocks[b].att.out_proj.bias = assign(
            gpt.trf_blocks[b].att.out_proj.bias, 
            params["blocks"][b]["attn"]["c_proj"]["b"])

        gpt.trf_blocks[b].ff.layers[0].weight = assign(
            gpt.trf_blocks[b].ff.layers[0].weight, 
            params["blocks"][b]["mlp"]["c_fc"]["w"].T)
        gpt.trf_blocks[b].ff.layers[0].bias = assign(
            gpt.trf_blocks[b].ff.layers[0].bias, 
            params["blocks"][b]["mlp"]["c_fc"]["b"])
        gpt.trf_blocks[b].ff.layers[2].weight = assign(
            gpt.trf_blocks[b].ff.layers[2].weight, 
            params["blocks"][b]["mlp"]["c_proj"]["w"].T)
        gpt.trf_blocks[b].ff.layers[2].bias = assign(
            gpt.trf_blocks[b].ff.layers[2].bias, 
            params["blocks"][b]["mlp"]["c_proj"]["b"])

        gpt.trf_blocks[b].norm1.scale = assign(
            gpt.trf_blocks[b].norm1.scale, 
            params["blocks"][b]["ln_1"]["g"])
        gpt.trf_blocks[b].norm1.shift = assign(
            gpt.trf_blocks[b].norm1.shift, 
            params["blocks"][b]["ln_1"]["b"])
        gpt.trf_blocks[b].norm2.scale = assign(
            gpt.trf_blocks[b].norm2.scale, 
            params["blocks"][b]["ln_2"]["g"])
        gpt.trf_blocks[b].norm2.shift = assign(
            gpt.trf_blocks[b].norm2.shift, 
            params["blocks"][b]["ln_2"]["b"])

    gpt.final_norm.scale = assign(gpt.final_norm.scale, params["g"])
    gpt.final_norm.shift = assign(gpt.final_norm.shift, params["b"])
    gpt.out_head.weight = assign(gpt.out_head.weight, params["wte"])
    
    
load_weights_into_gpt(gpt, params)
gpt.to(device);

- If the model is loaded correctly, we can use it to generate new text using our previous `generate` function:

In [ ]:
torch.manual_seed(123)

token_ids = generate(
    model=gpt,
    idx=text_to_token_ids("Every effort moves you", tokenizer).to(device),
    max_new_tokens=25,
    context_size=NEW_CONFIG["context_length"],
    top_k=50,
    temperature=1.5
)

print("Output text:\n", token_ids_to_text(token_ids, tokenizer))

- We know that we loaded the model weights correctly because the model can generate coherent text; if we made even a small mistake, the mode would not be able to do that

- For an alternative way to load the weights from the Hugging Face Hub, see [../02_alternative_weight_loading](../02_alternative_weight_loading)

## Summary and takeaways

- See the [./gpt_train.py](./gpt_train.py) script, a self-contained script for training
- The [./gpt_generate.py](./gpt_generate.py) script loads pretrained weights from OpenAI and generates text based on a prompt
- You can find the exercise solutions in [./exercise-solutions.ipynb](./exercise-solutions.ipynb)